# Tag-Bubble OCR Auto-Labeler (Colab GPU)

Runs EasyOCR across the 26K HVAC crops and produces `tag_bubble_labels.jsonl`.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run cells top to bottom. Upload buttons will appear where needed.
3. Expected runtime on T4: **1–3 hours**. Browser tab must stay open.

In [ ]:
# 1) Verify GPU
!nvidia-smi

In [ ]:
# 2) Upload label_tag_bubbles_ocr.py
from google.colab import files
print('Upload label_tag_bubbles_ocr.py')
up = files.upload()
assert 'label_tag_bubbles_ocr.py' in up, 'Please upload the file named label_tag_bubbles_ocr.py'

In [ ]:
# 3) Upload tag_dataset.zip (580 MB — takes several minutes)
from google.colab import files
print('Upload tag_dataset.zip (580 MB)')
up = files.upload()
assert 'tag_dataset.zip' in up, 'Please upload the file named tag_dataset.zip'

In [ ]:
# 4) Unzip dataset
import os, time
os.makedirs('tag_dataset', exist_ok=True)
t0 = time.time()
!unzip -q -o tag_dataset.zip -d tag_dataset
print(f'Unzipped in {time.time()-t0:.1f}s')
!find tag_dataset/images -name '*.png' | wc -l

In [ ]:
# 5) Install EasyOCR
!pip install -q easyocr
import easyocr
print('easyocr ok')

In [ ]:
# 6) Run the labeler on GPU. --resume lets you restart safely if the session drops.
!python -u label_tag_bubbles_ocr.py --dataset tag_dataset --out tag_bubble_labels.jsonl --gpu --resume

In [ ]:
# 7) Show summary + download the result
import json
from collections import Counter
from google.colab import files as gfiles

c = Counter()
for line in open('tag_bubble_labels.jsonl'):
    c[json.loads(line)['reason']] += 1
total = sum(c.values())
print()
for k, v in c.most_common():
    print(f'  {k:12s} {v:6d} ({100*v/total:.1f}%)')
print(f'  TOTAL: {total}')
print()
print('Download starting...')
gfiles.download('tag_bubble_labels.jsonl')